# 19.6 Fairness definitions & metrics

Fairness metrics compare rates across groups; the hard part is choosing which rate matches the harm.

Demographic parity uses all group members, equal opportunity uses positives only, and equalized odds checks both TPR and FPR gaps.

Save a copy to Drive to edit.

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.datasets import load_breast_cancer
from sklearn.datasets import make_blobs
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(19)


def clf_ladder():
    """D1..D5 classification ladder of rising complexity. Returns [(name, X, y), ...]."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def fit_ladder_classifier(X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr_s = scaler.fit_transform(x_tr)
    x_te_s = scaler.transform(x_te)
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_tr_s, y_tr)
    return clf, scaler, x_tr_s, x_te_s, y_tr, y_te


def score_for_class(model, X, class_index=None):
    proba = model.predict_proba(X)
    if class_index is None:
        class_index = int(np.argmax(proba[0]))
    return proba[:, class_index]


def finite_gradient(fn, x, eps=1e-4):
    grad = np.zeros_like(x, dtype=float)
    for j in range(len(x)):
        plus = x.copy()
        minus = x.copy()
        plus[j] = plus[j] + eps
        minus[j] = minus[j] - eps
        grad[j] = (fn(plus) - fn(minus)) / (2.0 * eps)
    return grad


def rank_corr(a, b):
    ar = pd.Series(a).rank(method="average").to_numpy()
    br = pd.Series(b).rank(method="average").to_numpy()
    if np.std(ar) == 0.0 or np.std(br) == 0.0:
        return 0.0
    return float(np.corrcoef(ar, br)[0, 1])


def top_k_mask(values, k):
    order = np.argsort(-np.abs(values))
    mask = np.zeros(len(values), dtype=bool)
    mask[order[:k]] = True
    return mask


## The concept, built once (D1)

Demographic parity is $\Delta_{DP}=|P(\hat Y=1\mid A=0)-P(\hat Y=1\mid A=1)|$. Equalized odds adds TPR and FPR gaps with their own denominators.

In [ ]:

def safe_rate(num, den):
    if den == 0:
        return np.nan
    return num / den


def fairness_report(y, yhat, group):
    report = {}
    for g in [0, 1]:
        mask = group == g
        positives = y == 1
        negatives = y == 0
        pred_pos = yhat == 1
        report[f"selection_{g}"] = safe_rate(np.sum(pred_pos & mask), np.sum(mask))
        report[f"tpr_{g}"] = safe_rate(np.sum(pred_pos & positives & mask), np.sum(positives & mask))
        report[f"fpr_{g}"] = safe_rate(np.sum(pred_pos & negatives & mask), np.sum(negatives & mask))
    report["dp_gap"] = abs(report["selection_0"] - report["selection_1"])
    report["tpr_gap"] = abs(report["tpr_0"] - report["tpr_1"])
    report["fpr_gap"] = abs(report["fpr_0"] - report["fpr_1"])
    report["eo_gap"] = max(report["tpr_gap"], report["fpr_gap"])
    return report

y_toy = np.array([1] * 20 + [0] * 20 + [1] * 20 + [0] * 20)
group_toy = np.array([0] * 40 + [1] * 40)
yhat_toy = np.array([1] * 13 + [0] * 7 + [1] * 13 + [0] * 7 + [1] * 9 + [0] * 11 + [1] * 9 + [0] * 11)
report_toy = fairness_report(y_toy, yhat_toy, group_toy)
total_toy = 0.65 + 0.45 + 0.2
abs_toy = total_toy
share_toy = 0.65 / abs_toy
guarded_toy = total_toy + 0.2 * abs_toy
contrast_toy = total_toy - 0.2

assert np.isclose(report_toy["selection_0"], 0.65)
assert np.isclose(report_toy["selection_1"], 0.45)
assert np.isclose(report_toy["dp_gap"], 0.2)
assert np.isclose(total_toy, 1.3)
assert np.isclose(abs_toy, 1.3)
assert np.isclose(share_toy, 0.5)
assert np.isclose(guarded_toy, 1.56)
assert np.isclose(contrast_toy - total_toy, -0.2)

print(report_toy)


For the ladder we induce an audit group from the median of the first standardized feature, then compute DP, TPR, FPR, and equalized-odds gaps on held-out predictions.

In [ ]:

def ladder_fairness(model, x_te, y_te):
    proba = model.predict_proba(x_te)
    if len(np.unique(y_te)) > 2:
        positive_class = int(np.unique(y_te)[-1])
        y_bin = (y_te == positive_class).astype(int)
        yhat = (np.argmax(proba, axis=1) == positive_class).astype(int)
    else:
        positive_class = 1
        y_bin = (y_te == positive_class).astype(int)
        yhat = (proba[:, positive_class] >= 0.5).astype(int)
    group = (x_te[:, 0] > np.median(x_te[:, 0])).astype(int)
    report = fairness_report(y_bin, yhat, group)
    return report, group, y_bin, yhat


## The dataset ladder

We use the shared F15 classification ladder: a hand linear toy, clean blobs, a nonlinear moons stress test, real Wine, and real Breast Cancer. The same logistic base model and the same metric code run unchanged across rungs.

In [ ]:

rungs = clf_ladder()

for index, item in enumerate(rungs, start=1):
    name, X, y = item
    counts = dict(zip(*np.unique(y, return_counts=True)))
    print(index, name)
    print("shape", X.shape)
    print("class counts", counts)
    print("sample", np.round(X[:3, : min(5, X.shape[1])], 3))


## Run the SAME method across D1-D5

The metric is fairness gap: primary demographic-parity gap, with equalized-odds components shown beside it.

In [ ]:

fair_rows = []
fair_artifacts = []

for rung, item in enumerate(rungs, start=1):
    name, X, y = item
    model, scaler, x_tr, x_te, y_tr, y_te = fit_ladder_classifier(X, y)
    report, group, y_bin, yhat = ladder_fairness(model, x_te, y_te)
    fair_rows.append({"rung": rung, "name": name, "dp_gap": report["dp_gap"], "tpr_gap": report["tpr_gap"], "fpr_gap": report["fpr_gap"], "eo_gap": report["eo_gap"]})
    fair_artifacts.append((name, report))

fair_table = pd.DataFrame(fair_rows)
print(fair_table.to_string(index=False))


## Results visualization

Panels show selection, TPR, and FPR by group. The curve tracks the demographic-parity gap under ladder stress.

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
axes = axes.ravel()

for ax, artifact in zip(axes[:5], fair_artifacts):
    name, report = artifact
    labels = ["sel0", "sel1", "tpr0", "tpr1", "fpr0", "fpr1"]
    values = [report["selection_0"], report["selection_1"], report["tpr_0"], report["tpr_1"], report["fpr_0"], report["fpr_1"]]
    ax.bar(labels, values)
    ax.set_ylim(0.0, 1.05)
    ax.set_title(name[:26])
    ax.tick_params(axis="x", rotation=45)

axes[5].plot(fair_table["rung"], fair_table["dp_gap"], marker="o", label="DP")
axes[5].plot(fair_table["rung"], fair_table["eo_gap"], marker="x", label="EO")
axes[5].set_title("fairness gap vs stress")
axes[5].set_xlabel("rung")
axes[5].set_ylabel("gap")
axes[5].set_ylim(0.0, 1.05)
axes[5].legend()
plt.tight_layout()


## Pitfall on the hardest rung

Pitfall: mixing denominators makes demographic parity and equal opportunity look interchangeable. The fix prints every numerator, denominator, and subgroup confusion matrix.

In [ ]:

name, X, y = rungs[-1]
model, scaler, x_tr, x_te, y_tr, y_te = fit_ladder_classifier(X, y)
report, group, y_bin, yhat = ladder_fairness(model, x_te, y_te)
wrong_gap = abs(np.mean(yhat[group == 0]) - np.mean(yhat[(group == 1) & (y_bin == 1)]))
fixed_gap = report["dp_gap"]

print("wrong mixed-denominator gap", wrong_gap)
print("fixed demographic-parity gap", fixed_gap)
for g in [0, 1]:
    mask = group == g
    tp = int(np.sum((yhat == 1) & (y_bin == 1) & mask))
    fp = int(np.sum((yhat == 1) & (y_bin == 0) & mask))
    tn = int(np.sum((yhat == 0) & (y_bin == 0) & mask))
    fn = int(np.sum((yhat == 0) & (y_bin == 1) & mask))
    print("group", g, "tp", tp, "fp", fp, "tn", tn, "fn", fn)


## Evaluate it + Practice

- Compare the reported DP and equalized-odds gaps with a no-skill baseline such as shuffled features, shuffled groups, or random rankings.
- Sanity check signs, denominators, and the background/reference point before trusting a pretty plot.
- Ablation: replace group-specific denominators with one pooled denominator and inspect which rate becomes invalid
- Failure signals: unstable ranks under small perturbations, a metric that improves while accuracy collapses, or explanations that change when the seed changes.
- For gap topics, especially influence functions, keep the lesson numbers visible and treat the notebook as an audit scaffold until the lesson body grows more examples.

Practice 1: Change the trust knob or kernel width and predict which metric should move before running it.

Practice 2: Swap the D5 background/reference set and explain which conclusion is no longer stable.

Practice 3: Turn the pitfall back on, then write one sentence explaining why the fixed version is safer.